# Traffic Sign Recognition with DCNN (Deep Convolutional Neural Networks)

This notebook presents a complete pipeline for building, training, and evaluating a Deep Convolutional Neural Network (DCNN) for multi-class traffic sign classification.

The project uses **TensorFlow/Keras** on the **GTSRB (German Traffic Sign Recognition Benchmark)** dataset, containing over 50,000 images across 43 classes.

### Architecture and Key Design Choices:
1. **Network architecture:** Custom `TrafficSignNet` model inspired by the classic AlexNet design. It uses 5 convolutional layers (Conv2D), pooling layers (MaxPooling2D), batch normalization, and fully connected (Dense) layers with a Softmax classifier.
2. **Image preprocessing:** Resolution standardized to $32 \times 32$ pixels. **CLAHE** (Contrast Limited Adaptive Histogram Equalization) compensates for uneven illumination and improves edge visibility before network input.
3. **Data augmentation:** On-the-fly image distortions (rotation, scaling, translation, shearing) to artificially expand the training set and reduce overfitting.
4. **Class imbalance compensation:** Class weights minimize the impact of underrepresented traffic sign categories on overall model accuracy.

In [ ]:
# Import required ML/CV libraries
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import BatchNormalization, Conv2D, MaxPooling2D, Activation, Flatten, Dropout, Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report
from skimage import transform, exposure, io
from imutils import paths
import imutils

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import random, os, glob, pickle, time, cv2

start_time = time.time()

class TrafficSignNet:
    @staticmethod
    def build(width, height, depth, classes):
        """
        Build the convolutional neural network graph.
        Layer configuration: (CONV => RELU => BN => POOL) + 2x (CONV => RELU => CONV => RELU => BN => POOL) + 2x FC.
        """
        model = Sequential()
        inputShape = (height, width, depth)
        chanDim = -1

        model.add(Input(shape=inputShape))

        # Block 1
        model.add(Conv2D(8, (5, 5), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=chanDim))
        model.add(MaxPooling2D(pool_size=(2, 2)))

        # Block 2
        model.add(Conv2D(16, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=chanDim))
        model.add(Conv2D(16, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=chanDim))
        model.add(MaxPooling2D(pool_size=(2, 2)))

        # Block 3
        model.add(Conv2D(32, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=chanDim))
        model.add(Conv2D(32, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=chanDim))
        model.add(MaxPooling2D(pool_size=(2, 2)))

        # Classification module (fully connected)
        model.add(Flatten())
        model.add(Dense(128))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.5))

        model.add(Flatten())
        model.add(Dense(128))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.5))

        # Softmax output (class probability distribution)
        model.add(Dense(classes))
        model.add(Activation("softmax"))

        return model

print("[SYSTEM] Model architecture compiled successfully.")

In [ ]:
def load_split(basePath, csvPath):
    """
    Extract and preprocess the dataset from reference CSV files.
    Applies adaptive histogram equalization (CLAHE) to normalize shading.
    """
    data = []
    labels = []
    rows = open(csvPath).read().strip().split("\n")[1:]
    random.shuffle(rows)

    for (i, row) in enumerate(rows):
        if i > 0 and i % 5000 == 0:
            print(f"[PREPROCESSING] Processed {i} images...")

        label, imagePath = row.strip().split(",")[-2:]
        imagePath = os.path.sep.join([basePath, imagePath])
        image = io.imread(imagePath)
        
        # Standardize input tensor resolution and normalize contrast (CLAHE)
        image = transform.resize(image, (32, 32))
        image = exposure.equalize_adapthist(image, clip_limit=0.1)
        
        data.append(image)
        labels.append(int(label))

    return (np.array(data), np.array(labels))

print("[SYSTEM] Data extraction utilities ready.")

In [ ]:
import gdown

# Automatic GTSRB dataset download for Google Colab environments
if not os.path.exists("signnames.csv"):
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/14_TSR_DCNN/signnames.csv --no-check-certificate

if not os.path.exists("gtsrb.zip"):
    print("[NETWORK] Downloading GTSRB repository (~300MB)...")
    gdown.download(id='1EQ-tyVHIdVaa4_1bob1zv8zgaVmvyyqV', output='gtsrb.zip', quiet=False)

if not os.path.exists('gtsrb'):
    print("[SYSTEM] Extracting archive...")
    !unzip -q 'gtsrb.zip'

dataset = "gtsrb/"
labelNames = open("signnames.csv").read().strip().split("\n")[1:]
labelNames = [l.split(",")[0] for l in labelNames]

trainPath = os.path.sep.join([dataset, "Train.csv"])
testPath = os.path.sep.join([dataset, "Test.csv"])

print("[INFO] Loading training and test matrices...")
(trainX, trainY) = load_split(dataset, trainPath)
(testX, testY) = load_split(dataset, testPath)

# Scale pixel values to [0.0, 1.0]
trainX = trainX.astype("float32") / 255.0
testX = testX.astype("float32") / 255.0

# One-hot encode labels for Softmax classifier
numLabels = len(np.unique(trainY))
trainY = to_categorical(trainY, numLabels)
testY = to_categorical(testY, numLabels)

# Serialize cached data to pickle for faster subsequent runs
with open('train_test.pickle', 'wb') as f:
    pickle.dump([trainX, trainY, testX, testY], f)

print("[INFO] Data cache saved successfully.")

In [ ]:
# --- MODEL HYPERPARAMETERS ---
NUM_EPOCHS = 15
INIT_LR = 1e-3
BS = 64

with open('train_test.pickle', 'rb') as f:
    trainX, trainY, testX, testY = pickle.load(f)

# Data augmentation configuration (simulates real-world noise)
aug = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.15,
    horizontal_flip=False, # Horizontal flip breaks semantic meaning of traffic signs
    vertical_flip=False,
    fill_mode="nearest"
)

# Exponential decay learning-rate schedule with Adam optimizer
print("[INFO] Compiling model...")
lr_schedule = ExponentialDecay(
    initial_learning_rate=INIT_LR,
    decay_steps=10000,
    decay_rate=INIT_LR / (NUM_EPOCHS * 0.5)
)
opt = Adam(learning_rate=lr_schedule)

model = TrafficSignNet.build(width=32, height=32, depth=3, classes=numLabels)
model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])

# Recompute class weights to compensate for GTSRB class frequency imbalance
classTotals = trainY.sum(axis=0)
classWeight = classTotals.max() / classTotals
classWeightD = {x: classWeight[x] for x in range(0, classWeight.shape[0])}

print("[INFO] Neural network ready for training.")

In [ ]:
# Start machine learning training loop (GPU acceleration recommended)
print("[INFO] Initializing network training...")
H = model.fit(
    aug.flow(trainX, trainY, batch_size=BS),
    validation_data=(testX, testY),
    epochs=NUM_EPOCHS,
    class_weight=classWeightD,
    verbose=1
)

print("[INFO] Serializing trained model to 'trafficsignnet.keras'...")
model.save("trafficsignnet.keras")
print("[INFO] Training complete.")

## Model Evaluation (Classification Metrics)

Model performance is verified on the held-out test set. Full statistical metrics help identify overfitting:
- **Precision:** Share of correctly classified signs (TP) among all predictions assigned to a class (TP + FP).
- **Recall (Sensitivity):** Ability to find all true instances of a class (TP / TP + FN).
- **F1-Score:** Harmonic mean of precision and recall; a rigorous overall classifier stability metric.

In [ ]:
print("[INFO] Estimating precision metrics...")
predictions = model.predict(testX, batch_size=BS)
print(classification_report(testY.argmax(axis=1), predictions.argmax(axis=1), target_names=labelNames))

# Generate learning curves
N = np.arange(0, NUM_EPOCHS)
plt.style.use("ggplot")
plt.figure(figsize=(10, 6))
plt.plot(N, H.history["loss"], label="Loss - Training")
plt.plot(N, H.history["val_loss"], label="Loss - Validation")
plt.plot(N, H.history["accuracy"], label="Accuracy - Training")
plt.plot(N, H.history["val_accuracy"], label="Accuracy - Validation")
plt.title("DCNN Training Progress (Loss & Accuracy)")
plt.xlabel("Epoch")
plt.ylabel("Loss / Accuracy")
plt.legend(loc="center right")
plt.savefig("train_plot.png")
plt.show()

## Inference (Real-World Testing)
Uses frozen weights from the serialized network to run in-the-wild tests on 25 randomly sampled test images, overlaying text predictions directly on the image (OpenCV OSD).

In [ ]:
print("[INFO] Loading parameters from disk...")
model = load_model("trafficsignnet.keras")

print("[INFO] Initializing inference...")
imagePaths = list(paths.list_images("gtsrb/Test"))
random.shuffle(imagePaths)
imagePaths = imagePaths[:25]

if not os.path.exists('examples'):
    os.mkdir('examples')

for (i, imagePath) in enumerate(imagePaths):
    image = io.imread(imagePath)
    image = transform.resize(image, (32, 32))
    image = exposure.equalize_adapthist(image, clip_limit=0.1)
    image = image.astype("float32") / 255.0
    image = np.expand_dims(image, axis=0)

    # Forward pass through the neural network
    preds = model.predict(image, verbose=0)
    j = preds.argmax(axis=1)[0]
    label = labelNames[j]

    # Build analytical output with text rendering (OSD)
    out_image = cv2.imread(imagePath)
    out_image = imutils.resize(out_image, width=128)
    cv2.putText(out_image, label, (5, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 2)
    p = os.path.sep.join(["examples", f"{i}.png"])
    cv2.imwrite(p, out_image)

print("[INFO] Inference complete.")
print(f"[SYSTEM] Total operational time: {time.time() - start_time:.2f} s")

In [ ]:
# Aggregate and visualize inference results
path = "examples/*.png"
images = []
for img_path in glob.glob(path):
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append(img)

fig = plt.figure(figsize=(20, 20))
plt.suptitle("TrafficSignNet Inference Grid", fontsize=24, y=0.92)
for i in range(len(images)):
    plt.subplot(5, 5, i + 1)
    plt.imshow(images[i])
    plt.axis('off')
plt.show()